# 03 - Model Training, Tuning & Evaluation
Trains 7 classification algorithms, tunes the top 3 with GridSearchCV, evaluates the best model, and saves it for production use.

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier

from src.evaluation import (plot_confusion_matrix, plot_roc_curve, plot_precision_recall_curve,
                             plot_learning_curve, get_cross_val_scores, plot_feature_importance,
                             get_classification_report)
from src.train_model import ModelTrainer, get_baseline_models, tune_model
from src.utils import MODELS_DIR, save_object

## Load engineered features

In [2]:
df = pd.read_csv('../data/telco_churn_engineered.csv')
X = df.drop(columns=['Churn'])
y = df['Churn']
print(X.shape, y.shape)

(7043, 53) (7043,)


## Train/Test Split (80/20, stratified, random_state=42)

In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y)
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (5634, 53) | Test: (1409, 53)


## Train Baseline Models
Logistic Regression, Decision Tree, Random Forest, KNN, SVM, Gradient Boosting, XGBoost

In [4]:
trainer = ModelTrainer()
models = get_baseline_models()
comparison_df = trainer.train_and_evaluate(models, X_train, y_train, X_test, y_test)
comparison_df.round(4)

2026-07-13 07:02:58 | INFO     | src.train_model | Training Logistic Regression ...


2026-07-13 07:02:59 | INFO     | src.train_model | Logistic Regression -> ROC AUC: 0.8467


2026-07-13 07:02:59 | INFO     | src.train_model | Training Decision Tree ...


2026-07-13 07:02:59 | INFO     | src.train_model | Decision Tree -> ROC AUC: 0.6596


2026-07-13 07:02:59 | INFO     | src.train_model | Training Random Forest ...


...

2026-07-13 07:03:11 | INFO     | src.train_model | Gradient Boosting -> ROC AUC: 0.8427


2026-07-13 07:03:11 | INFO     | src.train_model | Training XGBoost ...


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [07:03:11] WARNING: /__w/xgboost/xgboost/src/learner.cc:793: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


2026-07-13 07:03:11 | INFO     | src.train_model | XGBoost -> ROC AUC: 0.8195


,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.8077,0.6711,0.5401,0.5985,0.8467
Gradient Boosting,0.8013,0.6621,0.5134,0.5783,0.8427
SVM,0.7771,0.7273,0.2567,0.3794,0.8252
KNN,0.7928,0.6235,0.5535,0.5864,0.8221
Random Forest,0.7814,0.6071,0.5000,0.5484,0.8215
XGBoost,0.7708,0.5785,0.5027,0.5379,0.8195
Decision Tree,0.7239,0.4816,0.5241,0.5019,0.6596


## Hyperparameter Tuning (GridSearchCV)
Tuning Random Forest, Decision Tree, and Gradient Boosting.

In [5]:
rf_grid = {'n_estimators': [200, 300], 'max_depth': [8, 12, None], 'min_samples_split': [2, 5]}
best_rf, rf_params = tune_model(RandomForestClassifier(random_state=42), rf_grid, X_train, y_train)
print('Best RF params:', rf_params)

2026-07-13 07:03:11 | INFO     | src.train_model | Starting GridSearchCV for RandomForestClassifier ...


2026-07-13 07:04:34 | INFO     | src.train_model | Best params for RandomForestClassifier: {'max_depth': 8, 'min_samples_split': 5, 'n_estimators': 300} (CV roc_auc: 0.8444)


Best RF params: {'max_depth': 8, 'min_samples_split': 5, 'n_estimators': 300}


In [6]:
dt_grid = {'max_depth': [4, 6, 8, 10], 'min_samples_split': [2, 5, 10], 'criterion': ['gini', 'entropy']}
best_dt, dt_params = tune_model(DecisionTreeClassifier(random_state=42), dt_grid, X_train, y_train)
print('Best DT params:', dt_params)

2026-07-13 07:04:34 | INFO     | src.train_model | Starting GridSearchCV for DecisionTreeClassifier ...


2026-07-13 07:04:39 | INFO     | src.train_model | Best params for DecisionTreeClassifier: {'criterion': 'entropy', 'max_depth': 4, 'min_samples_split': 2} (CV roc_auc: 0.8277)


Best DT params: {'criterion': 'entropy', 'max_depth': 4, 'min_samples_split': 2}


In [7]:
gb_grid = {'n_estimators': [100, 200], 'learning_rate': [0.05, 0.1], 'max_depth': [2, 3, 4]}
best_gb, gb_params = tune_model(GradientBoostingClassifier(random_state=42), gb_grid, X_train, y_train)
print('Best GB params:', gb_params)

2026-07-13 07:04:39 | INFO     | src.train_model | Starting GridSearchCV for GradientBoostingClassifier ...


2026-07-13 07:06:32 | INFO     | src.train_model | Best params for GradientBoostingClassifier: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100} (CV roc_auc: 0.8481)


Best GB params: {'learning_rate': 0.05, 'max_depth': 2, 'n_estimators': 100}


## Compare tuned models against baselines

In [8]:
tuned_trainer = ModelTrainer()
tuned_models = {'Random Forest (Tuned)': best_rf, 'Decision Tree (Tuned)': best_dt,
                'Gradient Boosting (Tuned)': best_gb}
tuned_comparison = tuned_trainer.train_and_evaluate(tuned_models, X_train, y_train, X_test, y_test)
full_comparison = pd.concat([comparison_df, tuned_comparison]).sort_values('ROC AUC', ascending=False)
full_comparison.round(4)

2026-07-13 07:06:32 | INFO     | src.train_model | Training Random Forest (Tuned) ...


2026-07-13 07:06:33 | INFO     | src.train_model | Random Forest (Tuned) -> ROC AUC: 0.8425


2026-07-13 07:06:33 | INFO     | src.train_model | Training Decision Tree (Tuned) ...


2026-07-13 07:06:33 | INFO     | src.train_model | Decision Tree (Tuned) -> ROC AUC: 0.8299


2026-07-13 07:06:33 | INFO     | src.train_model | Training Gradient Boosting (Tuned) ...


2026-07-13 07:06:35 | INFO     | src.train_model | Gradient Boosting (Tuned) -> ROC AUC: 0.8427


,Accuracy,Precision,Recall,F1 Score,ROC AUC
Logistic Regression,0.8077,0.6711,0.5401,0.5985,0.8467
Gradient Boosting (Tuned),0.7977,0.6606,0.4893,0.5622,0.8427
Gradient Boosting,0.8013,0.6621,0.5134,0.5783,0.8427
Random Forest (Tuned),0.8013,0.6632,0.5107,0.5770,0.8425
Decision Tree (Tuned),0.7921,0.6576,0.4519,0.5357,0.8299
SVM,0.7771,0.7273,0.2567,0.3794,0.8252
KNN,0.7928,0.6235,0.5535,0.5864,0.8221
Random Forest,0.7814,0.6071,0.5000,0.5484,0.8215
XGBoost,0.7708,0.5785,0.5027,0.5379,0.8195
Decision Tree,0.7239,0.4816,0.5241,0.5019,0.6596


In [9]:
best_model_name = full_comparison.index[0]
all_fitted = {**trainer.fitted_models_, **tuned_trainer.fitted_models_}
best_model = all_fitted[best_model_name]
print('Best model:', best_model_name)
full_comparison.loc[best_model_name].round(4)

Best model: Logistic Regression


Accuracy     0.8077
Precision    0.6711
Recall       0.5401
F1 Score     0.5985
ROC AUC      0.8467
Name: Logistic Regression, dtype: float64

## Model Evaluation

In [10]:
y_pred = best_model.predict(X_test)
plot_confusion_matrix(y_test, y_pred)
print(get_classification_report(y_test, y_pred))

2026-07-13 07:06:35 | INFO     | src.evaluation | Saved confusion matrix to /home/claude/Customer_Churn_Project/outputs/plots/confusion_matrix.png


              precision    recall  f1-score   support

    No Churn       0.84      0.90      0.87      1035
       Churn       0.67      0.54      0.60       374

    accuracy                           0.81      1409
   macro avg       0.76      0.72      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [11]:
plot_roc_curve(best_model, X_test, y_test)

2026-07-13 07:06:35 | INFO     | src.evaluation | Saved ROC curve to /home/claude/Customer_Churn_Project/outputs/plots/roc_curve.png


PosixPath('/home/claude/Customer_Churn_Project/outputs/plots/roc_curve.png')

In [12]:
plot_precision_recall_curve(best_model, X_test, y_test)

2026-07-13 07:06:35 | INFO     | src.evaluation | Saved precision-recall curve to /home/claude/Customer_Churn_Project/outputs/plots/precision_recall_curve.png


PosixPath('/home/claude/Customer_Churn_Project/outputs/plots/precision_recall_curve.png')

In [13]:
plot_learning_curve(best_model, X_train, y_train)

2026-07-13 07:06:42 | INFO     | src.evaluation | Saved learning curve to /home/claude/Customer_Churn_Project/outputs/plots/learning_curve.png


PosixPath('/home/claude/Customer_Churn_Project/outputs/plots/learning_curve.png')

In [14]:
cv_scores = get_cross_val_scores(best_model, X_train, y_train)
print('Mean CV ROC AUC:', cv_scores.mean().round(4))

2026-07-13 07:06:44 | INFO     | src.evaluation | Cross-val roc_auc scores: [0.8674, 0.8621, 0.8528, 0.8428, 0.8238] | Mean: 0.8498


Mean CV ROC AUC: 0.8498


## Feature Importance (Top 20)

In [15]:
importance_source = best_model if hasattr(best_model, 'feature_importances_') else best_rf
imp_df = plot_feature_importance(importance_source, list(X_train.columns), top_n=20)
imp_df

/home/claude/Customer_Churn_Project/notebooks/../src/evaluation.py:151: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=imp_df, x="Importance", y="Feature", ax=ax, palette="viridis")


2026-07-13 07:06:45 | INFO     | src.evaluation | Saved feature importance chart to /home/claude/Customer_Churn_Project/outputs/plots/feature_importance.png


,Feature,Importance
25,Contract_Month-to-month,0.098609
4,tenure,0.091124
9,TotalCharges,0.066789
14,TotalCharges_log,0.066284
8,MonthlyCharges,0.059769
50,TechSupport_No,0.057610
10,AvgMonthlySpend,0.055436
47,OnlineSecurity_No,0.052471
29,InternetService_Fiber optic,0.052023
23,TenureGroup_New Customer,0.040550


## Save the Best Model

In [16]:
save_object(best_model, MODELS_DIR / 'best_model.pkl')
print('Model saved to models/best_model.pkl')

Model saved to models/best_model.pkl


## Summary
The Logistic Regression model achieved the best ROC AUC (~0.85) on the held-out test set, closely followed by the tuned Gradient Boosting and Random Forest models. Given the moderate class imbalance, ROC AUC and F1 score are the most trustworthy metrics for model selection. See the project README for full business insights and recommendations.